# Gold Layer — Full Schema Build

Builds all **reference dimensions** and **fact tables** from the silver layer.

## Dimensions
| Step | Table | Source | PK |
|------|-------|--------|----|
| 1 | `dim_date` | Generated (2015–2035) | `date_key` (YYYYMMDD int) |
| 2 | `dim_department` | silver.departments | `dept_sk` (int) |
| 3 | `dim_medication` | silver.medications | `medication_sk` (int) |
| 4 | `dim_diagnosis_code` | silver.diagnoses (ICD-10 codes) | `diagnosis_sk` (int) |
| 5 | `dim_payer` | silver.insurance_claims | `payer_sk` (int; 0 = SELF-PAY) |

## Fact Tables
| Step | Table | Source | Grain | Surrogate PK |
|------|-------|--------|-------|--------------|
| 6 | `fact_appointments` | silver.encounters (OUTPATIENT/EMERGENCY) | encounter_id | appointment_sk |
| 7 | `fact_admissions` | silver.admissions | admission_id | admission_sk |
| 8 | `fact_medical_records` | silver.encounters + silver.diagnoses | encounter_id | medical_record_sk |
| 9 | `fact_surgeries` | silver.procedures | procedure_id | surgery_sk |
| 10 | `fact_lab_results` | silver.lab_results + silver.lab_orders | result_id | lab_result_sk |
| 11 | `fact_prescriptions` | silver.prescriptions | prescription_id | prescription_sk |
| 12 | `fact_pharmacy_dispenses` | silver.pharmacy_transactions | transaction_id | dispense_sk |
| 13 | `fact_billing` | silver.invoices + payments + insurance_claims | invoice_id | billing_sk |

> **dim_patient** and **dim_doctor** are built by the identity-resolution
> notebooks (01 & 02) and are NOT re-built here.

## 0. (Optional) Teardown — Drop All Gold Tables Before Rebuild

Run the cell below **only** when doing a full clean rebuild.  
Skip if doing an incremental refresh.

In [ ]:
# ⚠️  DESTRUCTIVE — Uncomment and run only for a full pipeline rebuild.
# Do NOT drop dim_patient_enterprise or dim_doctor_enterprise unless
# you also plan to re-run both identity-resolution notebooks.

# teardown_sql = [
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_billing",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_pharmacy_dispenses",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_prescriptions",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_lab_results",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_surgeries",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_medical_records",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_admissions",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.fact_appointments",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.dim_payer",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.dim_diagnosis_code",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.dim_medication",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.dim_department",
#     "DROP TABLE IF EXISTS hospital_catalog.gold.dim_date",
# ]
# for sql in teardown_sql:
#     print(f'Running: {sql}')
#     spark.sql(sql)
# print('\n✅ Teardown complete.')

print('Teardown skipped (cells commented out).')

## 1. Setup

In [ ]:
# No external library installs required.
# PySpark and Delta Lake are natively provided by Databricks Runtime.
print('Environment ready.')

In [ ]:
import sys
sys.path.insert(0, '/Workspace/Users/sallam5100@gmail.com/.bundle/hospital_edw/dev/files')

In [ ]:
from config.gold_config import GOLD_TABLES, SILVER_TABLES, BRIDGE_TABLES, DATE_SPINE_START, DATE_SPINE_END

print('Gold tables :')
for k, v in GOLD_TABLES.items():
    print(f'  {k:<30} → {v}')
print()
print('Bridge maps:')
for k, v in BRIDGE_TABLES.items():
    print(f'  {k:<30} → {v}')

## 2. Verify Prerequisites

Confirm the identity-resolution bridge maps exist before building facts.

In [ ]:
from src.gold.writer import table_exists

prereqs = {
    'dim_patient':  GOLD_TABLES['dim_patient'],
    'dim_doctor':   GOLD_TABLES['dim_doctor'],
    'patient_map':  BRIDGE_TABLES['patient_map'],
    'doctor_map':   BRIDGE_TABLES['doctor_map'],
}

all_ok = True
for name, tbl in prereqs.items():
    exists = table_exists(spark, tbl)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {name}: {tbl}')
    if not exists:
        all_ok = False

if not all_ok:
    raise RuntimeError(
        'One or more prerequisite tables are missing. '
        'Run the identity-resolution notebooks first.'
    )
print('\nAll prerequisites satisfied ✅')

## 3. Quick Silver Layer Check

Row counts from all silver source tables used by the gold builder.

In [ ]:
print(f"{'Silver Table':<35} {'Rows':>10}")
print('-' * 47)
for name, tbl in SILVER_TABLES.items():
    try:
        n = spark.table(tbl).count()
        print(f"  {name:<33} {n:>10,}")
    except Exception as e:
        print(f"  {name:<33} {'ERROR':>10}  ({e})")

## 4. Build Reference Dimensions

Steps 1–5: dim_date, dim_department, dim_medication, dim_diagnosis_code, dim_payer.  
Each dimension now carries an **integer surrogate key** as its primary key.

In [ ]:
from src.gold.dimensions import build_all_dimensions

build_all_dimensions(
    spark,
    tables        = GOLD_TABLES,
    silver_tables = SILVER_TABLES,
    date_start    = DATE_SPINE_START,
    date_end      = DATE_SPINE_END,
)

## 5. Build Fact Tables

Steps 6–13 (all 8 fact tables including the new pharmacy facts).  
Each fact table now carries an integer `<name>_sk` surrogate key as its row identifier.

In [ ]:
from src.gold.facts import build_all_facts

build_all_facts(
    spark,
    tables        = GOLD_TABLES,
    silver_tables = SILVER_TABLES,
    bridge_tables = BRIDGE_TABLES,
)

## 6. Validate — Row Counts & Spot Checks

In [ ]:
from pyspark.sql import functions as F

# ── Dimension counts ──────────────────────────────────────────────
dim_keys = [
    'dim_date', 'dim_patient', 'dim_doctor',
    'dim_department', 'dim_medication', 'dim_diagnosis_code', 'dim_payer'
]
print('DIMENSIONS')
print(f"  {'Table':<30} {'Rows':>10}")
print('  ' + '-' * 42)
for k in dim_keys:
    tbl = GOLD_TABLES[k]
    n = spark.table(tbl).count()
    print(f"  {k:<30} {n:>10,}")

# ── Fact counts ───────────────────────────────────────────────────
fact_keys = [
    'fact_appointments', 'fact_admissions', 'fact_medical_records',
    'fact_surgeries', 'fact_lab_results',
    'fact_prescriptions', 'fact_pharmacy_dispenses',
    'fact_billing'
]
print()
print('FACTS')
print(f"  {'Table':<30} {'Rows':>10}")
print('  ' + '-' * 42)
for k in fact_keys:
    tbl = GOLD_TABLES[k]
    n = spark.table(tbl).count()
    print(f"  {k:<30} {n:>10,}")

In [ ]:
# ── Payer breakdown — verify SELF-PAY is not the only row ─────────
display(
    spark.table(GOLD_TABLES['dim_payer'])
    .orderBy('payer_sk')
)

In [ ]:
# ── Billing: payer_sk distribution (0 = SELF-PAY, >0 = insured) ──
display(
    spark.table(GOLD_TABLES['fact_billing'])
    .groupBy('payer_sk', 'payer_name')
    .agg(
        F.count('*').alias('invoice_count'),
        F.round(F.sum('gross_amount'), 2).alias('gross_total'),
        F.round(F.sum('total_amount_paid'), 2).alias('collected_total'),
        F.round(F.sum('outstanding_balance'), 2).alias('outstanding_total'),
        F.round(F.sum('total_insurance_paid'), 2).alias('insurance_paid_total'),
    )
    .orderBy(F.desc('invoice_count'))
)

In [ ]:
# ── Billing: payment method breakdown ────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_billing'])
    .groupBy('payment_method')
    .agg(F.count('*').alias('invoice_count'))
    .orderBy(F.desc('invoice_count'))
)

In [ ]:
# ── Pharmacy: top 10 dispensed medications ────────────────────────
display(
    spark.table(GOLD_TABLES['fact_pharmacy_dispenses'])
    .join(
        spark.table(GOLD_TABLES['dim_medication'])
        .select('medication_sk', 'medication_name', 'category'),
        on='medication_sk',
        how='left',
    )
    .groupBy('medication_sk', 'medication_name', 'category')
    .agg(
        F.count('*').alias('dispense_count'),
        F.sum('quantity').alias('total_units'),
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
    )
    .orderBy(F.desc('dispense_count'))
    .limit(10)
)

In [ ]:
# ── Pharmacy: payment method split ───────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_pharmacy_dispenses'])
    .groupBy('payment_method')
    .agg(
        F.count('*').alias('txn_count'),
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
    )
    .orderBy(F.desc('txn_count'))
)

In [ ]:
# ── Surgeries: top procedures by volume ──────────────────────────
display(
    spark.table(GOLD_TABLES['fact_surgeries'])
    .groupBy('procedure_name')
    .agg(
        F.count('*').alias('surgery_count'),
        F.round(F.avg('duration_minutes'), 1).alias('avg_duration_min'),
    )
    .orderBy(F.desc('surgery_count'))
    .limit(15)
)

In [ ]:
# ── Lab results: abnormal rate by test ───────────────────────────
display(
    spark.table(GOLD_TABLES['fact_lab_results'])
    .groupBy('test_name')
    .agg(
        F.count('*').alias('total_results'),
        F.sum(F.col('is_abnormal').cast('int')).alias('abnormal_count'),
        F.round(
            F.sum(F.col('is_abnormal').cast('int')) * 100.0 / F.count('*'), 1
        ).alias('abnormal_pct'),
    )
    .orderBy(F.desc('total_results'))
)

In [ ]:
# ── Admissions: LOS distribution ─────────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_admissions'])
    .filter(F.col('length_of_stay_days').isNotNull())
    .groupBy('admission_type')
    .agg(
        F.count('*').alias('admissions'),
        F.round(F.avg('length_of_stay_days'), 1).alias('avg_los_days'),
        F.max('length_of_stay_days').alias('max_los_days'),
    )
    .orderBy(F.desc('admissions'))
)

## 7. Referential Integrity Checks

Verify that every fact row has a valid FK in the dimension tables.

In [ ]:
def ri_check(fact_tbl, fact_col, dim_tbl, dim_col, label):
    fact = spark.table(fact_tbl)
    dim  = spark.table(dim_tbl).select(dim_col).distinct()
    orphans = fact.join(dim, fact[fact_col] == dim[dim_col], how='left_anti')
    n = orphans.count()
    status = '✅' if n == 0 else f'⚠️  {n:,} orphans'
    print(f'  {status}  {label}')

print('Referential integrity:')
# Patient FK
for fact in ['fact_appointments', 'fact_admissions', 'fact_medical_records',
             'fact_surgeries', 'fact_lab_results',
             'fact_prescriptions', 'fact_pharmacy_dispenses', 'fact_billing']:
    ri_check(GOLD_TABLES[fact], 'enterprise_patient_id',
             GOLD_TABLES['dim_patient'], 'enterprise_patient_id',
             f'{fact} → dim_patient')

print()
# Dept FK (facts that carry dept_sk)
for fact in ['fact_appointments', 'fact_admissions', 'fact_medical_records']:
    ri_check(GOLD_TABLES[fact], 'dept_sk',
             GOLD_TABLES['dim_department'], 'dept_sk',
             f'{fact} → dim_department')

print()
# Payer FK
ri_check(GOLD_TABLES['fact_billing'], 'payer_sk',
         GOLD_TABLES['dim_payer'], 'payer_sk',
         'fact_billing → dim_payer')

print()
# Medication FK
for fact in ['fact_prescriptions', 'fact_pharmacy_dispenses']:
    ri_check(GOLD_TABLES[fact], 'medication_sk',
             GOLD_TABLES['dim_medication'], 'medication_sk',
             f'{fact} → dim_medication')

print()
# Diagnosis FK
ri_check(GOLD_TABLES['fact_medical_records'], 'diagnosis_sk',
         GOLD_TABLES['dim_diagnosis_code'], 'diagnosis_sk',
         'fact_medical_records → dim_diagnosis_code')

## 8. Star Schema Summary

```
                    dim_date
                       │  (date_key → *_date_key)
                       │
dim_patient ─────── fact_appointments ─────── dim_doctor
                         │  dept_sk ──────── dim_department

dim_patient ─────── fact_admissions ──────── dim_department

dim_patient ─────── fact_medical_records ─── dim_doctor
                         │  dept_sk ──────── dim_department
                         │  diagnosis_sk ─── dim_diagnosis_code

dim_patient ─────── fact_surgeries ──────── dim_doctor (surgeon)

dim_patient ─────── fact_lab_results ────── dim_doctor (ordering)

dim_patient ─────── fact_prescriptions ──── dim_doctor
                         │  medication_sk ─ dim_medication

dim_patient ─────── fact_pharmacy_dispenses  dim_doctor
                         │  medication_sk ─ dim_medication

dim_patient ─────── fact_billing ─────────── dim_payer
```

Connect Power BI directly to `hospital_catalog.gold.*`.